In [13]:
!uv pip install geopy

Using Python 3.12.3 environment at: /home/camarada/venv/lighthouse/.venv
Resolved 2 packages in 302ms                                         
⠙ Preparing packages... (0/2)                                                   
⠙ Preparing packages... (0/2)-------------------     0 B/122.50 KiB          
⠙ Preparing packages... (0/2)------------------- 14.93 KiB/122.50 KiB        
⠙ Preparing packages... (0/2)------------------- 30.93 KiB/122.50 KiB        
⠙ Preparing packages... (0/2)------------------- 46.93 KiB/122.50 KiB        
⠙ Preparing packages... (0/2)--------------- 62.93 KiB/122.50 KiB        
⠙ Preparing packages... (0/2)----------- 78.93 KiB/122.50 KiB        
⠙ Preparing packages... (0/2)---------- 94.93 KiB/122.50 KiB        
⠙ Preparing packages... (0/2)---------- 94.93 KiB/122.50 KiB        
geographiclib        ------------------------------     0 B/39.79 KiB
⠙ Preparing packages... (0/2)---------- 94.93 KiB/122.50 KiB        
geographiclib        ----------------------

## Geoencode Cities

In [ ]:
from geopy.geocoders import Nominatim
import numpy as np
import pandas as pd
import time

In [54]:
## load the csv with the cities
dfcity = pd.read_csv("mod_national_capitals.csv")

In [83]:
cols = ['query_city', 'query_country','name','display_name',
        'lat','lon','boundingbox','osm_id']


## create a geolocator object
geolocator = Nominatim(user_agent="capital_geocoder")

df_geolocated = pd.DataFrame()

for index, row in dfcity.iterrows():
    sample = f"{row['capital']}, {row['name_query']}"
    print(sample)
    time.sleep(2)

    location = None

    try:
        location = geolocator.geocode(sample)
    except Exception as e:
        print(f"Geocoding error for: {sample}")
        print(e)

    if location:
        location.raw['boundingbox'] = [location.raw['boundingbox']]
        dff = pd.DataFrame.from_dict(location.raw, orient='columns')
    else:
        dff = pd.DataFrame(
            np.nan,
            index=[0],
            columns=[
                'place_id', 'licence', 'osm_type', 'osm_id',
                'lat', 'lon', 'class', 'type', 'place_rank',
                'importance', 'addresstype', 'name',
                'display_name', 'boundingbox'
            ]
        )
        print(f"Geocoding failed for: {sample}")

    # align indices before concat
    df_conc = pd.concat(
        [
            dff.reset_index(drop=True),
            pd.DataFrame(row).T.reset_index(drop=True)
        ],
        axis=1
    )

    ## concat to the final df
    df_geolocated = pd.concat(
        [df_geolocated, df_conc],
        ignore_index=True
    )

## save df
df_geolocated.to_csv("geolocated_capitals.csv", index=False)

Andorra la Vella, Andorra
Tirana, Albania
Yerevan, Armenia
Luanda, Angola
Buenos Aires, Argentina
Calafate, Argentina
Salzburg, Austria
Canberra, Australia
Gold Coast, Australia
Brussels, Belgium
Sofia, Bulgaria
Sucre, Bolivia
Brasilia, Brazil
Florianopolis, Brazil
Minsk, Belarus
Ottawa, Canada
Kinshasa, Democratic Republic of the Congo
Brazzaville, Republic of the Congo
Santiago, Chile
Beijing, People's Republic of China
Bogota, Colombia
San Jose, Costa Rica
Havana, Cuba
Nicosia, Cyprus
Prague, Czech Republic
Berlin, Germany
Munich, Germany
Copenhagen, Denmark
Santo Domingo, Dominican Republic
Quito, Ecuador
Tallinn, Estonia
Cairo, Egypt
Addis Ababa, Ethiopia
Helsinki, Finland
Suva, Fiji
Rennes, France
Bordeaux, France
Paris, France
Tbilisi, Georgia
Athens, Greece
Guatemala City, Guatemala
Budapest, Hungary
Jakarta, Indonesia
Dublin, Republic of Ireland
Jerusalem, Israel
New Delhi, India
Geocoding error for: New Delhi, India
HTTPSConnectionPool(host='nominatim.openstreetmap.org', port

In [84]:
df_geolocated.shape

(119, 18)